# Task Dependencies & Context (CrewAI)

This notebook accompanies [Agent Foundry](https://agent-foundry-pi.vercel.app).

In [ ]:
!pip install -q crewai crewai-tools

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## Explicit context and fan-in

The first crew uses **`context=[research_task]`** so the analyst only sees the researcher’s output. The second adds a **fan-in**: **`summary_task`** lists **both** `research_task` and `data_task` in `context`, so the writer merges two upstream deliverables.

In [ ]:
from crewai import Agent, Crew, Process, Task

researcher = Agent(
    role="Research Analyst",
    goal="Produce short factual bullets from the prompt only",
    backstory="You do not browse the web; you reason briefly and stay concrete.",
    verbose=True,
)
analyst = Agent(
    role="Analyst",
    goal="Turn research bullets into implications",
    backstory="You only react to the context you are given.",
    verbose=True,
)
writer = Agent(
    role="Writer",
    goal="Merge multiple inputs into one short brief",
    backstory="You reconcile two streams without inventing data.",
    verbose=True,
)

research_chain = Task(
    description="List three reasons teams adopt agent frameworks (reasoning only, no tools).",
    expected_output="Exactly three bullets, one line each.",
    agent=researcher,
)

analysis_task = Task(
    description="From the research context, state one risk and one opportunity.",
    expected_output="Two sentences: one risk, one opportunity.",
    agent=analyst,
    context=[research_chain],
)

crew_explicit = Crew(
    agents=[researcher, analyst],
    tasks=[research_chain, analysis_task],
    process=Process.sequential,
    verbose=True,
)
print("--- Explicit context (research → analysis) ---")
crew_explicit.kickoff()

research_fanin = Task(
    description="List three reasons teams adopt agent frameworks (reasoning only, no tools).",
    expected_output="Exactly three bullets, one line each.",
    agent=researcher,
)

data_task = Task(
    description="List three KPI-style metrics teams might track for agent deployments (made-up but plausible names).",
    expected_output="Three bullets: metric name + one phrase of meaning.",
    agent=analyst,
)

summary_task = Task(
    description="Combine the research bullets and the KPI bullets into six non-overlapping bullets under two subheadings.",
    expected_output="Markdown: two ## headings, six bullets total.",
    agent=writer,
    context=[research_fanin, data_task],
)

crew_fanin = Crew(
    agents=[researcher, analyst, writer],
    tasks=[research_fanin, data_task, summary_task],
    process=Process.sequential,
    verbose=True,
)
print("--- Fan-in (research + data → summary) ---")
crew_fanin.kickoff()

## `async_execution` and callbacks

Two tasks use **`async_execution=True`** so they can run together; **`synthesis_task`** uses **`context=[...]`** for both, so it starts only after **both** finish. **`callback`** prints when that merge task completes.

In [ ]:
from crewai import Agent, Crew, Process, Task
from crewai.tasks.task_output import TaskOutput


def on_complete(output: TaskOutput):
    print(f"Task done: {output.description}")


researcher = Agent(
    role="Research Analyst",
    goal="Short thematic bullets from the prompt only",
    backstory="Stay under four bullets per task; no tool use.",
    verbose=True,
)
synthesizer = Agent(
    role="Synthesis Lead",
    goal="Unify parallel inputs",
    backstory="You merge without dropping distinct points.",
    verbose=True,
)

research_llms = Task(
    description="Three bullets on LLM adoption themes in products.",
    expected_output="Three bullets.",
    agent=researcher,
    async_execution=True,
)

research_agents = Task(
    description="Three bullets on multi-agent orchestration themes.",
    expected_output="Three bullets.",
    agent=researcher,
    async_execution=True,
)

synthesis_task = Task(
    description="Merge both research outputs into six bullets under two ## headings.",
    expected_output="Markdown with two ## headings and six bullets.",
    agent=synthesizer,
    context=[research_llms, research_agents],
    callback=on_complete,
)

crew = Crew(
    agents=[researcher, synthesizer],
    tasks=[research_llms, research_agents, synthesis_task],
    process=Process.sequential,
    verbose=True,
)

result = crew.kickoff()
print(result)

## Key takeaways

- **`context=[upstream, ...]`** pins which task outputs the model sees for the current task.
- **Fan-in** is a single task whose `context` lists **multiple** prior tasks.
- **`async_execution=True`** on several tasks runs them **concurrently**; a task that **depends** on them via `context` **waits for all**.
- **`callback`** runs on completion and receives a **`TaskOutput`** (here logged with `output.description`).